# CC-Delta Sweep Results Analysis

This notebook analyzes the results of CC-Delta steering experiments, comparing
different intervention methods (SAE-based CC-Delta, dense CAA, prompt defense)
on their **safety vs. utility** trade-off.

CC-Delta uses sparse autoencoder (SAE) features to steer language models away from
harmful outputs while preserving general capabilities. See the
[README](../README.md) for full details.

## How to run experiments

Before using this notebook, run the experiments. The easiest way is:

```bash
# Run all stages (baseline, CC-Delta, CAA) in fast mode
python -m ccdelta.scripts.run_experiments

# Run on the full dataset
python -m ccdelta.scripts.run_experiments --full

# Run specific stages only
python -m ccdelta.scripts.run_experiments --stages baseline ccdelta
```

See the [README](../README.md) for more options.

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd

from notebooks.notebook_utils import (
    add_safety_score,
    add_utility,
    expand_generation_params,
    find_optimal_configs,
    group_mean_scores,
    load_sweep_results,
    plot_multi_metric,
)

In [ ]:
# --- Configuration: update these paths to match your experiment layout ---
# These match the defaults from `python -m ccdelta.scripts.run_experiments`

RESULTS_ROOT = Path("./artifacts")  # Change to "./results" for default run_experiments output

BASELINE_DIR = RESULTS_ROOT / "baseline"
PROMPT_DEFENSE_DIR = RESULTS_ROOT / "prompt_defense"
SAE_SWEEP_ROOT = RESULTS_ROOT / "sae"
DENSE_SWEEP_ROOT = RESULTS_ROOT / "dense"

# Utility metrics to aggregate (uses whatever is available)
UTILITY_METRICS = ["fluency", "ifeval", "mmlu"]

In [ ]:
# --- Load all experiments ---

dfs = []

sources = [
    (BASELINE_DIR, "baseline"),
    (PROMPT_DEFENSE_DIR, "prompt_defense"),
    (SAE_SWEEP_ROOT, "sae"),
    (DENSE_SWEEP_ROOT, "dense"),
]

for path, label in sources:
    if not path.exists():
        print(f"WARNING: {path} not found, skipping {label}")
        continue
    try:
        df = load_sweep_results(path, method_label=label)
        dfs.append(df)
        print(f"Loaded {label}: {len(df)} rows")
    except FileNotFoundError as e:
        print(f"WARNING: {e}")

if not dfs:
    raise RuntimeError(
        "No experiment results found. Run experiments first (see instructions above)."
    )

all_results = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(all_results)} rows across {all_results['intervention_type'].nunique()} methods")

In [ ]:
# --- Compute metrics ---

all_results = expand_generation_params(all_results)
all_results["safety"] = add_safety_score(all_results)

# Unify CAA "strength" into "steering_multiplier" so methods are comparable
if "strength" in all_results.columns:
    all_results["steering_multiplier"] = all_results["steering_multiplier"].fillna(
        all_results["strength"]
    )

# Only compute utility for metrics that are actually present
available_metrics = [m for m in UTILITY_METRICS if m in all_results.columns]
if available_metrics:
    all_results = add_utility(all_results, available_metrics)
    print(f"Utility computed from: {available_metrics}")
else:
    print("WARNING: No utility metrics found. Utility column not created.")

all_results.head()

In [ ]:
# --- Aggregate per configuration ---

group_cols = ["intervention_type", "generation_param"]
if "model" in all_results.columns:
    group_cols.insert(0, "model")

score_cols = ["safety"]
if "utility" in all_results.columns:
    score_cols.append("utility")
for m in ["fluency", "ifeval", "mmlu"]:
    if m in all_results.columns:
        score_cols.append(m)

agg_results = group_mean_scores(all_results, group_cols=group_cols, score_cols=score_cols)
agg_results = expand_generation_params(agg_results)

# Unify strength → steering_multiplier in aggregated results too
if "strength" in agg_results.columns:
    agg_results["steering_multiplier"] = agg_results["steering_multiplier"].fillna(
        agg_results["strength"]
    )

print(f"Aggregated to {len(agg_results)} configurations")
agg_results

In [ ]:
# --- Safety vs Utility (all configurations) ---

if "utility" in agg_results.columns:
    tooltip_cols = [c for c in ["intervention_type", "steering_multiplier",
                                 "num_features", "safety", "utility",
                                 "fluency", "generation_param"]
                    if c in agg_results.columns]

    scatter = alt.Chart(agg_results).mark_circle(size=100).encode(
        x=alt.X("safety:Q", title="Safety"),
        y=alt.Y("utility:Q", title="Utility"),
        color=alt.Color("intervention_type:N", title="Method"),
        tooltip=tooltip_cols,
    ).properties(
        title="Safety vs Utility", width=500, height=350
    )

    if "steering_multiplier" in agg_results.columns:
        scatter = scatter.encode(
            size=alt.Size("steering_multiplier:Q", title="Steering Multiplier",
                          scale=alt.Scale(range=[40, 300])),
        )

    display(scatter)
else:
    print("Skipping: 'utility' column not available.")

In [ ]:
# --- Per-method metric breakdown (line charts) ---

metrics = [c for c in ["safety", "fluency", "ifeval", "mmlu", "utility"] if c in agg_results.columns]
baseline_df = agg_results[agg_results["intervention_type"] == "baseline"]

# CC-Delta (SAE): metrics vs steering_multiplier
sae_df = agg_results[agg_results["intervention_type"] == "sae"]
if len(sae_df) > 0 and "steering_multiplier" in sae_df.columns:
    display(plot_multi_metric(
        sae_df, metrics, x_col="steering_multiplier",
        reference_df=baseline_df, title="CC-Delta: Metrics vs Steering Multiplier",
    ))

# CAA (dense): metrics vs steering_multiplier (unified from strength)
dense_df = agg_results[agg_results["intervention_type"] == "dense"]
if len(dense_df) > 0 and "steering_multiplier" in dense_df.columns:
    display(plot_multi_metric(
        dense_df, metrics, x_col="steering_multiplier",
        reference_df=baseline_df, title="CAA: Metrics vs Steering Multiplier",
    ))

In [ ]:
# --- Best configurations per threshold ---
# Paper methodology: maximize safety subject to utility metrics not
# dropping more than threshold_change below baseline.

if "safety" in agg_results.columns:
    baseline_df = agg_results[agg_results["intervention_type"] == "baseline"].copy()
    intervention_df = agg_results[agg_results["intervention_type"] != "baseline"].copy()

    if len(baseline_df) > 0 and len(intervention_df) > 0:
        optimal = find_optimal_configs(
            intervention_df,
            baseline_df,
            threshold_changes=[0.05, 0.10, 0.25, 1.0],
            target_score="safety",
        )

        if len(optimal) > 0:
            display_cols = [c for c in ["intervention_type", "generation_param",
                                         "threshold_change", "safety", "safety_diff",
                                         "fluency", "fluency_diff",
                                         "ifeval", "ifeval_diff",
                                         "mmlu", "mmlu_diff",
                                         "utility", "sample_count"]
                            if c in optimal.columns]
            display(optimal[display_cols])
        else:
            print("No valid configurations found at any threshold level.")
    else:
        print("Need both baseline and intervention results.")
else:
    print("Skipping: safety column not available.")

In [ ]:
# --- Method comparison plot (paper-style) ---
# Plots optimal configs at each threshold alongside baseline,
# connected by lines per method.

if 'optimal' in dir() and len(optimal) > 0:
    # Build baseline row
    baseline_point = baseline_df.iloc[[0]].copy()
    baseline_point["threshold_change"] = 0.0

    # Combine optimal configs with baseline
    plot_cols = [c for c in ["intervention_type", "threshold_change",
                             "safety", "utility", "fluency", "ifeval", "mmlu",
                             "generation_param"]
                if c in optimal.columns]
    # Ensure baseline has same columns
    for c in plot_cols:
        if c not in baseline_point.columns:
            baseline_point[c] = None
    plot_df = pd.concat([baseline_point[plot_cols], optimal[plot_cols]], ignore_index=True)

    # Safety vs Utility
    if "utility" in plot_df.columns:
        points = alt.Chart(plot_df).mark_circle(size=80).encode(
            x=alt.X("safety:Q", title="Safety"),
            y=alt.Y("utility:Q", title="Utility", scale=alt.Scale(zero=False)),
            color=alt.Color("intervention_type:N", title="Method"),
            size=alt.Size("threshold_change:Q", title="Max Utility Loss",
                          scale=alt.Scale(range=[30, 200])),
            tooltip=[c for c in plot_cols if c in plot_df.columns],
        )

        lines = alt.Chart(plot_df).mark_line(
            strokeDash=[8, 2], opacity=0.7
        ).encode(
            x="safety:Q",
            y="utility:Q",
            color="intervention_type:N",
            detail="intervention_type:N",
        )

        chart = (lines + points).properties(
            title="Safety vs Utility (best configs per threshold)",
            width=200, height=200,
        )
        display(chart)

    # Safety vs individual metrics
    metric_charts = []
    for metric in ["fluency", "ifeval", "mmlu"]:
        if metric not in plot_df.columns:
            continue
        pts = alt.Chart(plot_df).mark_circle(size=60).encode(
            x=alt.X("safety:Q", title="Safety"),
            y=alt.Y(f"{metric}:Q", title=metric.replace('_', ' ').title(),
                     scale=alt.Scale(zero=False)),
            color=alt.Color("intervention_type:N", title="Method"),
            tooltip=[c for c in plot_cols if c in plot_df.columns],
        )
        ln = alt.Chart(plot_df).mark_line(
            strokeDash=[8, 2], opacity=0.7
        ).encode(
            x="safety:Q",
            y=f"{metric}:Q",
            color="intervention_type:N",
            detail="intervention_type:N",
        )
        metric_charts.append((ln + pts).properties(
            title=f"Safety vs {metric.title()}", width=150, height=150,
        ))

    if metric_charts:
        display(alt.hconcat(*metric_charts))
else:
    print("Run the optimal configs cell above first.")